# Phase 7B — Object Detection with YOLOv8 & torchvision

**Theory:** Object detection = classify AND localize objects (bounding boxes).
YOLO (You Only Look Once) processes the entire image in one forward pass — fast and accurate.

**Install:**
```
pip install ultralytics torch torchvision
```

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

ULTRALYTICS_AVAILABLE = False
TORCH_AVAILABLE = False

try:
    from ultralytics import YOLO

    ULTRALYTICS_AVAILABLE = True
    print("Ultralytics (YOLOv8) ready")
except ImportError:
    print("Install: pip install ultralytics")

try:
    import torch
    import torchvision

    TORCH_AVAILABLE = True
    print(f"PyTorch: {torch.__version__}, torchvision: {torchvision.__version__}")
except ImportError:
    print("Install: pip install torch torchvision")

---
## 1. YOLO Architecture Concepts

**How YOLO works:**
1. Divide image into S×S grid
2. Each cell predicts B bounding boxes + confidence + C class probabilities
3. Single forward pass = all predictions simultaneously
4. Non-Maximum Suppression (NMS) removes duplicate boxes

**Output per detection:**
- `(x, y, w, h)` — bounding box coordinates and dimensions
- `confidence` — how certain the model is
- `class_id` — what object was detected

In [ ]:
# Visualize what bounding box output looks like
fig, ax = plt.subplots(figsize=(8, 6))

# Simulate a simple scene
scene = np.ones((400, 600, 3), dtype=np.uint8) * 220

# Draw some colored rectangles to simulate objects
scene[50:150, 50:200] = [100, 149, 237]  # blue rectangle = person
scene[200:350, 300:500] = [255, 165, 0]  # orange = car
scene[100:200, 350:450] = [60, 179, 113]  # green = bicycle

ax.imshow(scene)

# Simulated YOLO detections
detections = [
    {"bbox": (50, 50, 150, 100), "label": "person", "conf": 0.92, "color": "blue"},
    {"bbox": (300, 200, 200, 150), "label": "car", "conf": 0.87, "color": "orange"},
    {"bbox": (350, 100, 100, 100), "label": "bicycle", "conf": 0.79, "color": "green"},
]

for det in detections:
    x, y, w, h = det["bbox"]
    rect = patches.Rectangle(
        (x, y), w, h, linewidth=2, edgecolor=det["color"], facecolor="none"
    )
    ax.add_patch(rect)
    ax.text(
        x,
        y - 5,
        f"{det['label']} {det['conf']:.0%}",
        color=det["color"],
        fontsize=10,
        fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.7),
    )

ax.set_title("Object Detection Output — Bounding Boxes + Labels + Confidence")
ax.axis("off")
plt.tight_layout()
plt.show()

---
## 2. YOLOv8 — Usage

In [ ]:
if ULTRALYTICS_AVAILABLE:
    # Load pretrained YOLOv8 nano (smallest, fastest)
    # First run downloads the model (~6MB)
    model = YOLO("yolov8n.pt")  # n=nano, s=small, m=medium, l=large, x=xlarge

    print("Model loaded:")
    print(f"  Task: {model.task}")
    print(f"  Classes: {len(model.names)} COCO classes")
    print(f"  Example classes: {list(model.names.values())[:10]}")
else:
    print("YOLOv8 not installed. Run: pip install ultralytics")
    print()
    print("Once installed, usage is:")
    print("  from ultralytics import YOLO")
    print("  model = YOLO('yolov8n.pt')")
    print("  results = model('path/to/image.jpg')")
    print("  results[0].show()")

In [ ]:
if ULTRALYTICS_AVAILABLE:
    import urllib.request
    import os

    # Download a test image
    test_img_url = "https://ultralytics.com/images/bus.jpg"
    test_img_path = "test_bus.jpg"

    if not os.path.exists(test_img_path):
        urllib.request.urlretrieve(test_img_url, test_img_path)
        print(f"Downloaded: {test_img_path}")

    # Run inference
    results = model(test_img_path, conf=0.5)  # conf=confidence threshold

    # Display results
    result = results[0]
    print(f"\nDetections: {len(result.boxes)}")
    for box in result.boxes:
        cls_id = int(box.cls)
        conf = float(box.conf)
        xyxy = box.xyxy[0].tolist()  # (x1, y1, x2, y2)
        print(
            f"  {model.names[cls_id]:12s}  conf={conf:.2f}  box={[int(c) for c in xyxy]}"
        )

    # Plot with bounding boxes
    annotated = result.plot()
    plt.figure(figsize=(10, 6))
    plt.imshow(annotated[:, :, ::-1])  # BGR → RGB
    plt.axis("off")
    plt.title("YOLOv8 Detection Results")
    plt.show()

---
## 3. torchvision — Datasets and Pretrained Models

In [ ]:
if TORCH_AVAILABLE:
    import torchvision.transforms as T
    from torchvision import models
    from PIL import Image
    import torch

    # Standard ImageNet transform pipeline — use this for any pretrained model
    transform = T.Compose(
        [
            T.Resize(256),
            T.CenterCrop(224),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )
    print("Transform pipeline for ImageNet models:")
    print(transform)

    # Available pretrained models in torchvision
    print("\nPopular pretrained models:")
    model_names = [
        "resnet18",
        "resnet50",
        "vgg16",
        "efficientnet_b0",
        "mobilenet_v3_small",
    ]
    for name in model_names:
        model_fn = getattr(models, name)
        m = model_fn()
        params = sum(p.numel() for p in m.parameters())
        print(f"  {name:25s}: {params / 1e6:.1f}M parameters")

---
## 4. IoU — Intersection Over Union

The core metric for object detection. Measures how well a predicted bounding box overlaps with the ground truth.

In [ ]:
def compute_iou(box1, box2):
    """Compute IoU between two boxes in (x1, y1, x2, y2) format."""
    x1_inter = max(box1[0], box2[0])
    y1_inter = max(box1[1], box2[1])
    x2_inter = min(box1[2], box2[2])
    y2_inter = min(box1[3], box2[3])

    inter_area = max(0, x2_inter - x1_inter) * max(0, y2_inter - y1_inter)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union_area = area1 + area2 - inter_area

    return inter_area / union_area if union_area > 0 else 0.0


# Example
ground_truth = (100, 100, 300, 250)
prediction_good = (110, 105, 295, 245)  # close
prediction_bad = (200, 200, 400, 350)  # offset

iou_good = compute_iou(ground_truth, prediction_good)
iou_bad = compute_iou(ground_truth, prediction_bad)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, pred, iou, title in zip(
    axes,
    [prediction_good, prediction_bad],
    [iou_good, iou_bad],
    ["Good Prediction", "Bad Prediction"],
):
    ax.set_xlim(0, 450)
    ax.set_ylim(0, 400)
    ax.invert_yaxis()

    gt_rect = patches.Rectangle(
        (ground_truth[0], ground_truth[1]),
        ground_truth[2] - ground_truth[0],
        ground_truth[3] - ground_truth[1],
        lw=3,
        edgecolor="green",
        facecolor="green",
        alpha=0.3,
        label="Ground Truth",
    )
    pr_rect = patches.Rectangle(
        (pred[0], pred[1]),
        pred[2] - pred[0],
        pred[3] - pred[1],
        lw=3,
        edgecolor="red",
        facecolor="red",
        alpha=0.3,
        label="Prediction",
    )
    ax.add_patch(gt_rect)
    ax.add_patch(pr_rect)
    ax.set_title(f"{title}\nIoU = {iou:.3f}")
    ax.legend()

plt.suptitle("Intersection over Union (IoU)", fontsize=13)
plt.tight_layout()
plt.show()

print("IoU thresholds: >0.5 = acceptable, >0.75 = good, >0.9 = excellent")

---
## Summary

| Library | Purpose |
|---------|--------|
| Pillow | Load, save, basic transforms (resize, crop, rotate) |
| OpenCV | Image processing (filters, edge detection, contours, video) |
| torchvision | Datasets, pretrained CNNs, augmentation pipeline |
| ultralytics | YOLOv8 — detection, segmentation, pose estimation |

**YOLOv8 model sizes:**
| Model | Size | Speed | Accuracy |
|-------|------|-------|----------|
| yolov8n | 6MB | Fastest | Lowest |
| yolov8s | 22MB | Fast | |
| yolov8m | 49MB | Medium | |
| yolov8l | 83MB | Slower | |
| yolov8x | 130MB | Slowest | Highest |